# Native ApET: repeat question 2058428

A small repeatability check, with one historically stable GQA control. Enable Internet and **GPU T4 ×2**. Native ApET, dependencies, checkpoints, prompts and seeds remain fixed. Default: **72 recorded generations, one model load**, plus four short warmups. Read `research/GQA_REPEATABILITY.md` for interpretation.

The first flag is False to stop automatic execution before setup/inference. Open this notebook in the editor and run cells in order when ready.

## 1. Update code and choose a new run directory


In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

RUN_REPEATABILITY_TEST = False  # Change to True when ready.
MODEL_SESSIONS = 1  # Optional: 2 for two sequential fresh model processes.
REPEATS = 3
PREPARED_MANIFEST = None  # Optional existing GQA pilot/development manifest path.
checkpoint_args = []  # Optional ["--model", local_llava, "--vision-model", local_clip].
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
if not RUN_REPEATABILITY_TEST:
    raise RuntimeError("Set RUN_REPEATABILITY_TEST=True to start this small check.")
if MODEL_SESSIONS not in {1, 2} or not 2 <= REPEATS <= 10:
    raise ValueError("Use one or two model sessions and 2–10 repeats")

repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path(f"/kaggle/working/original-gqa-repeatability-{stamp}")
run_dir.mkdir()
original_python = repo / "environments/original/.venv/bin/python"
report_paths = [run_dir / f"session-{i + 1}.json" for i in range(MODEL_SESSIONS)]
print("Run directory:", run_dir)
print("Recorded generations:", MODEL_SESSIONS * 2 * (4 * (REPEATS + 3) + 12))


## 2. Reuse/check the original runtime

Stop other model processes first. This reuses the isolated packages when available; a fresh Kaggle session downloads them again. Existing `/tmp` checkpoint caches are reused. The notebook kernel remains unchanged.


In [ ]:
subprocess.run([
    "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
    "--output", str(run_dir / "original-runtime.json"),
], cwd=repo, check=True)


## 3. Repeated generation, logging off/on and configuration switching

One subprocess per model session. Both inspected cases are fetched from pinned source files (~68 MB), or from `PREPARED_MANIFEST`. Only two images are saved. Each session warms up once, then performs consecutive repeats, logging sandwiches and A → B → A configuration controls. Do not enable new deterministic settings for this first check.

Answer changes are findings and do not abort the experiment. Execution failures preserve partial records. Interrupt this cell to stop the child process, then run the download cell.


In [ ]:
import signal

manifest_args = ["--data-manifest", str(PREPARED_MANIFEST)] if PREPARED_MANIFEST else []
for report_path in report_paths:
    command = [
        str(original_python), "-I", "scripts/repeat_original_gqa.py",
        "--repeats", str(REPEATS), "--download-budget-gib", "20",
        *manifest_args, *checkpoint_args, "--output", str(report_path),
    ]
    print("Starting:", report_path.name, flush=True)
    process = subprocess.Popen(command, cwd=repo)
    try:
        returncode = process.wait()
    except KeyboardInterrupt:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        print("Stopped. Run the final cell to download available evidence.")
        raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


## 4. Inspect and download

`passed` means execution completed; inspect the separate variation findings. A stable result here does not explain the historical disagreement. ZIP includes both images, question metadata, native arrays, raw answers, runtime/source snapshots and `uv.lock`. It excludes environments and model weights. This cell also packages partial evidence after interruption/failure.


In [ ]:
import zipfile
from IPython.display import FileLink, display

if "process" in globals() and process.poll() is None:
    raise RuntimeError("Stop the evaluator before creating the ZIP.")
reports = []
for report_path in report_paths:
    if not report_path.is_file():
        continue
    result = json.loads(report_path.read_text())
    reports.append(result)
    print(report_path.name, "status:", result["status"], "progress:", result["progress"])
    print("Findings:", result["findings"])
    if result.get("error"):
        print("Error:", result["error"])
    for qid, configurations in result.get("summaries", {}).items():
        print("Question:", qid)
        for name, summary in configurations.items():
            print(" ", name, summary["raw_answer_counts"], "stable:", summary["raw_answer_stable"],
                  "by logging:", summary["by_logging"])
if len(reports) == 2:
    comparisons = {}
    for qid in set(reports[0].get("summaries", {})) & set(reports[1].get("summaries", {})):
        comparisons[qid] = {}
        for name in set(reports[0]["summaries"][qid]) & set(reports[1]["summaries"][qid]):
            first, second = [report["summaries"][qid][name] for report in reports]
            comparisons[qid][name] = {
                "raw_answer_distributions_equal": first["raw_answer_counts"] == second["raw_answer_counts"],
                "logging_distributions_equal": first["by_logging"] == second["by_logging"],
            }
    comparison = {"both_sessions_complete": all(r["status"] == "passed" for r in reports),
                  "comparisons": comparisons,
                  "note": "Distribution comparison only; complete trial-level evidence is in each JSONL."}
    (run_dir / "session-comparison.json").write_text(json.dumps(comparison, indent=2) + "\n")
    print("Between-session comparison:", comparison)
bundle = run_dir.with_suffix(".zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir.rglob("*")):
        if path.is_file():
            archive.write(path, str(path.relative_to(run_dir)))
print("Download before stopping the session:", bundle)
display(FileLink(str(bundle)))
